# Offline Kaggle runner

1. Add the competition dataset `playground-series-s6e8` **and** the uploaded `kaggle_bundle-*.zip` dataset.
2. This notebook extracts the bundle, sets `SMARTPHONE_ADDICTION_ROOT`, installs the wheel with `--no-deps`, verifies the environment, then trains from the **bundle directory** with competition CSVs mounted under `/kaggle/input/playground-series-s6e8`.
3. Build a submission CSV from the completed run. Never auto-upload.

No model source code lives in this notebook — only CLI calls against the packaged wheel.


In [ ]:
from pathlib import Path

WORKING = Path("/kaggle/working")
BUNDLE = WORKING / "bundle"
DATA = Path("/kaggle/input/playground-series-s6e8")  # competition data mount
INPUT = Path("/kaggle/input")
print("input mounts:", sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else "missing")
print("competition data:", DATA, "exists=", DATA.exists())
if DATA.exists():
    print("csv files:", sorted(p.name for p in DATA.glob("*.csv")))

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

WORKING = Path("/kaggle/working")
BUNDLE = WORKING / "bundle"
zips = sorted(Path("/kaggle/input").rglob("kaggle_bundle-*.zip"))
assert zips, "Upload kaggle_bundle-*.zip as a Kaggle dataset and attach it to this notebook"
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)
with zipfile.ZipFile(zips[0]) as zf:
    zf.extractall(BUNDLE)
os.environ["SMARTPHONE_ADDICTION_ROOT"] = str(BUNDLE)
os.chdir(BUNDLE)
print("extracted", BUNDLE)
print("bundle files:", sorted(p.name for p in BUNDLE.iterdir())[:20])
print("cwd=", Path.cwd())
print("SMARTPHONE_ADDICTION_ROOT=", os.environ["SMARTPHONE_ADDICTION_ROOT"])

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BUNDLE = Path(os.environ["SMARTPHONE_ADDICTION_ROOT"])
os.chdir(BUNDLE)
whl = next(BUNDLE.glob("smartphone_addiction-*.whl"))
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", str(whl)])
subprocess.check_call([sys.executable, str(BUNDLE / "verify_environment.py")])
print("installed", whl.name)

In [ ]:
import os
import subprocess
from pathlib import Path

BUNDLE = Path(os.environ["SMARTPHONE_ADDICTION_ROOT"])
DATA = Path("/kaggle/input/playground-series-s6e8")
os.chdir(BUNDLE)
assert DATA.is_dir(), f"Attach competition data at {DATA}"

# Prefer the bundled launcher (sets ROOT + train args). DATA_DIR points at Kaggle input.
env = os.environ.copy()
env["DATA_DIR"] = str(DATA)
env["SMARTPHONE_ADDICTION_ROOT"] = str(BUNDLE)
launcher = BUNDLE / "run_offline.sh"
print("running", launcher, "with DATA_DIR=", DATA)
subprocess.check_call(["bash", str(launcher)], cwd=BUNDLE, env=env)

In [ ]:
import os
import subprocess
from pathlib import Path

BUNDLE = Path(os.environ["SMARTPHONE_ADDICTION_ROOT"])
DATA = Path("/kaggle/input/playground-series-s6e8")
os.chdir(BUNDLE)
runs = sorted((BUNDLE / "artifacts" / "runs").glob("*")) if (BUNDLE / "artifacts" / "runs").exists() else []
assert runs, "no completed runs under artifacts/runs"
run_dir = runs[-1]
print("building submission from", run_dir)
subprocess.check_call(
    [
        "smartphone-addiction",
        "submission",
        "build",
        "--run",
        str(run_dir),
        "--sample",
        str(DATA / "sample_submission.csv"),
        "--output",
        str(Path("/kaggle/working") / "submission.csv"),
    ],
    cwd=BUNDLE,
)
print("Wrote /kaggle/working/submission.csv — download manually. Never auto-upload.")